# DistilBERT on SST-2 Experiment Console

This notebook is the unified entry point for the repository. It only handles parameter configuration, stage-by-stage execution, result loading, and visualization. The training, evaluation, and token-importance analysis logic stays in importable Python modules under `src/model`.


## Environment And Dependency Check

Run this cell first to locate the project root, expose the package root on `sys.path`, and inspect the core package versions.


In [ ]:
import json
import sys
from pathlib import Path

import datasets
import pandas as pd
import plotly.express as px
import torch
import transformers
from IPython.display import HTML, Markdown, display


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model import AnalysisConfig, EvalConfig, TrainConfig, evaluate, run_attention_analysis, train

pd.set_option("display.max_colwidth", 160)

versions_df = pd.DataFrame(
    [
        {"package": "python", "version": sys.version.split()[0]},
        {"package": "torch", "version": torch.__version__},
        {"package": "transformers", "version": transformers.__version__},
        {"package": "datasets", "version": datasets.__version__},
        {"package": "pandas", "version": pd.__version__},
    ]
)
display(versions_df)
print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")


## Configuration

Edit the three config objects below. Their field boundaries are intentionally strict:

- `TrainConfig` contains training-only settings.
- `EvalConfig` contains evaluation-only settings.
- `AnalysisConfig` contains token-importance-analysis-only settings.

To skip training and reuse an existing checkpoint, leave the training cell unexecuted and run only the evaluation or analysis cells.


In [ ]:
train_config = TrainConfig(
    model_name="distilbert-base-uncased",
    cache_dir=".cache/huggingface",
    output_dir="results/checkpoints/distilbert_sst2",
    metrics_dir="results/metrics/distilbert_sst2",
    max_length=128,
    seed=42,
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=2.0,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    logging_steps=50,
    save_total_limit=2,
    metric_for_best_model="accuracy",
    max_train_examples=None,
    max_eval_examples=None,
)

eval_config = EvalConfig(
    checkpoint_dir=train_config.output_dir,
    cache_dir=train_config.cache_dir,
    metrics_dir=train_config.metrics_dir,
    split="validation",
    max_length=train_config.max_length,
    seed=train_config.seed,
    per_device_eval_batch_size=train_config.per_device_eval_batch_size,
    max_eval_examples=None,
)

analysis_config = AnalysisConfig(
    checkpoint_dir=train_config.output_dir,
    cache_dir=train_config.cache_dir,
    metrics_dir=train_config.metrics_dir,
    analysis_dir="results/analysis/distilbert_sst2",
    max_length=train_config.max_length,
    seed=train_config.seed,
    sample_size=200,
    analysis_subset="correct_only",
    top_k_values=[1, 2, 3],
    top_ratios=[],
    random_trials=5,
    max_validation_examples=None,
    case_study_count=5,
)

display(pd.DataFrame(
    [
        {"config": "train", "output_dir": train_config.output_dir, "metrics_dir": train_config.metrics_dir},
        {"config": "eval", "checkpoint_dir": eval_config.checkpoint_dir, "metrics_dir": eval_config.metrics_dir},
        {"config": "analysis", "checkpoint_dir": analysis_config.checkpoint_dir, "analysis_dir": analysis_config.analysis_dir},
    ]
))


## Training Stage

Run this cell when you want to fine-tune the model.


In [ ]:
train_result = train(train_config)

## Evaluation Stage

Run this cell to evaluate a saved checkpoint on the selected split.


In [ ]:
eval_result = evaluate(eval_config)

## Importance Analysis Stage

Run this cell to produce token-importance rankings, ranking-similarity tables, deletion records, and qualitative cases.


In [ ]:
analysis_result = run_attention_analysis(analysis_config)

## Result Loading Helpers

The next helpers make the notebook robust when you rerun only selected stages.


In [ ]:
def load_json_if_exists(path: Path):
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_csv_if_exists(path: Path):
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    return pd.read_csv(path)


def display_markdown_file(path: Path):
    if not path.exists():
        print(f"Missing file: {path}")
        return
    display(Markdown(path.read_text(encoding="utf-8")))


artifact_paths = {
    "train_metrics": PROJECT_ROOT / train_config.metrics_dir / "train_metrics.json",
    "run_config": PROJECT_ROOT / train_config.metrics_dir / "run_config.json",
    "validation_metrics": PROJECT_ROOT / eval_config.metrics_dir / f"{eval_config.split}_metrics.json",
    "validation_predictions": PROJECT_ROOT / eval_config.metrics_dir / f"{eval_config.split}_predictions.csv",
    "ranking_similarity": PROJECT_ROOT / analysis_config.analysis_dir / "ranking_similarity_summary.csv",
    "deletion_summary": PROJECT_ROOT / analysis_config.analysis_dir / "deletion_summary.csv",
    "deletion_records": PROJECT_ROOT / analysis_config.analysis_dir / "deletion_records.csv",
    "qualitative_cases": PROJECT_ROOT / analysis_config.analysis_dir / "qualitative_cases.md",
}

display(pd.DataFrame([
    {"artifact": name, "path": str(path), "exists": path.exists()}
    for name, path in artifact_paths.items()
]))


## Validation Metrics And Prediction Sample


In [ ]:
validation_metrics = load_json_if_exists(artifact_paths["validation_metrics"])
if validation_metrics is not None:
    display(pd.DataFrame([validation_metrics]))

prediction_sample = load_csv_if_exists(artifact_paths["validation_predictions"])
if prediction_sample is not None:
    display(prediction_sample.head(10))


## Ranking Similarity


In [ ]:
ranking_similarity = load_csv_if_exists(artifact_paths["ranking_similarity"])
if ranking_similarity is not None:
    display(ranking_similarity)


## Deletion Summary And Confidence Drop Visualization


In [ ]:
deletion_summary = load_csv_if_exists(artifact_paths["deletion_summary"])
if deletion_summary is not None:
    display(deletion_summary)
    fig = px.line(
        deletion_summary,
        x="actual_k",
        y="mean_confidence_drop",
        color="method",
        markers=True,
        title="Confidence Drop After Token Deletion",
        labels={
            "actual_k": "Deleted token count",
            "mean_confidence_drop": "Mean confidence drop",
            "method": "Deletion method",
        },
    )
    fig.show()


## Qualitative Cases


In [ ]:
display_markdown_file(artifact_paths["qualitative_cases"])


## Optional Notes

- To reuse an existing checkpoint, update `eval_config.checkpoint_dir` and `analysis_config.checkpoint_dir`, then run only the evaluation and analysis cells.
- To rerun analysis only, change the fields in `analysis_config` and rerun `analysis_result = run_attention_analysis(analysis_config)`.
- `sample_size` now controls the subset used by attention, gradient × input, and leave-one-out comparison; `loo` remains the most expensive method.
- To run a smaller smoke test from the notebook, reduce `max_train_examples`, `max_eval_examples`, `sample_size`, and `max_validation_examples` before executing the stage cells.
